# SPCA comparison: Bair vs manifold optimization (proper train/test split)

This notebook compares two supervised PCA approaches across 5 regression datasets:
- **Bair's method** (feature selection + PCA + regression),
- **Manifold optimization SPCA** (Grassmann optimization with prediction + reconstruction loss).

We use a proper evaluation protocol:
1. Outer 80/20 train/test split (before any preprocessing).
2. Hyperparameters θ (Bair) and λ (manifold) are selected via an inner 80/20 split on the training set.
3. Models are refit on the full training set with the selected HPs.
4. Final metrics are reported on the held-out test set.

In [9]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import torch

from utils import (
    load_parkinsons_data,
    load_energy_efficiency_data,
    load_real_estate_data,
    load_wine_quality_data,
    load_student_performance_data,
    compare_bair_vs_manifold,
)

In [10]:
k = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

datasets = {
    "Parkinsons": lambda: load_parkinsons_data(device),
    "Student Performance": lambda: load_student_performance_data(device),
    "Energy Efficiency": lambda: load_energy_efficiency_data(device),
    "Real Estate Valuation": lambda: load_real_estate_data(device),
    "Wine Quality": lambda: load_wine_quality_data(device),
}

all_rows = []
for name, loader in datasets.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")

    result = loader()
    if len(result) == 3:
        X_raw, y_raw, _ = result
    else:
        X_raw, y_raw = result

    res = compare_bair_vs_manifold(X_raw, y_raw, k=k)

    for method in ["bair", "manifold"]:
        label = "Bair SPCA" if method == "bair" else "Manifold SPCA"
        all_rows.append({
            "Dataset": name,
            "Method": label,
            "Variance explained": res[method]["ve"],
            "Prediction error (MSE)": res[method]["pred_err"],
            "Runtime (sec)": res[method]["runtime"],
        })


Dataset: Parkinsons
theta = 0.100:  val_mse = 0.3445,  val_ve = 0.6944
theta = 0.358:  val_mse = 0.3445,  val_ve = 0.6944
theta = 0.616:  val_mse = 0.3445,  val_ve = 0.6101
theta = 0.874:  val_mse = 0.3445,  val_ve = 0.6101
theta = 1.132:  val_mse = 0.3435,  val_ve = 0.4011
theta = 1.389:  val_mse = 0.3398,  val_ve = 0.1919
theta = 1.647:  val_mse = 0.3381,  val_ve = 0.0990
theta = 1.905:  val_mse = 0.3381,  val_ve = 0.0990
theta = 2.163:  val_mse = 0.3426,  val_ve = 0.0859
theta = 2.421:  val_mse = 0.3426,  val_ve = 0.0859
theta = 2.679:  val_mse = 0.3426,  val_ve = 0.0859
theta = 2.937:  val_mse = 0.3426,  val_ve = 0.0859
theta = 3.195:  val_mse = 0.3426,  val_ve = 0.0859
theta = 3.453:  val_mse = 0.3426,  val_ve = 0.0859
theta = 3.711:  val_mse = 0.3426,  val_ve = 0.0859
theta = 3.968:  val_mse = 0.3422,  val_ve = 0.0855
theta = 4.226:  val_mse = 0.2812,  val_ve = 0.0404
theta = 4.484:  val_mse = 0.2812,  val_ve = 0.0404
theta = 4.742:  val_mse = 0.2949,  val_ve = 0.0353
theta = 5.

In [11]:
results = pd.DataFrame(all_rows)
results.set_index(["Method", "Dataset"], inplace=True)
results

,,Variance explained,Prediction error (MSE),Runtime (sec)
Method,Dataset,,,
Bair SPCA,Parkinsons,0.048665,0.297345,0.000427
Manifold SPCA,Parkinsons,0.771264,0.234256,2.275043
Bair SPCA,Student Performance,0.176264,0.096650,0.001277
Manifold SPCA,Student Performance,0.379390,0.086366,0.784797
Bair SPCA,Energy Efficiency,0.642677,0.083437,0.000200
Manifold SPCA,Energy Efficiency,0.641018,0.037595,1.340440
Bair SPCA,Real Estate Valuation,0.750574,0.092320,0.000160
Manifold SPCA,Real Estate Valuation,0.610381,0.051218,0.570533
Bair SPCA,Wine Quality,0.093952,0.127791,0.000226


In [12]:
pivot = results.reset_index().pivot(index="Dataset", columns="Method",
                                     values=["Variance explained", "Prediction error (MSE)", "Runtime (sec)"])
avg_row = pivot.mean().to_frame().T
avg_row.index = ["Average"]
pivot_with_avg = pd.concat([pivot, avg_row])
pivot_with_avg

Variance explained               Prediction error (MSE)  \
Method                         Bair SPCA Manifold SPCA              Bair SPCA   
Energy Efficiency               0.642677      0.641018               0.083437   
Parkinsons                      0.048665      0.771264               0.297345   
Real Estate Valuation           0.750574      0.610381               0.092320   
Student Performance             0.176264      0.379390               0.096650   
Wine Quality                    0.093952      0.778108               0.127791   
Average                         0.342426      0.636032               0.139508   

                                    Runtime (sec)                
Method                Manifold SPCA     Bair SPCA Manifold SPCA  
Energy Efficiency          0.037595      0.000200      1.340440  
Parkinsons                 0.234256      0.000427      2.275043  
Real Estate Valuation      0.051218      0.000160      0.570533  
Student Performance        0.086366      0.001277      0.784797  
Wine Quality               0.092083      0.000226      2.014379  
Average                    0.100304      0.000458      1.397038